### Backpropagation Implementation in Neural Networks

#### Backpropagation is the process of training a neural network by adjusting the weights based on the error between the predicted output and the actual output. It involves computing the gradient of the loss function with respect to each weight using the chain rule of calculus.
This allows the network to minimize the loss function and improve its predictions over time.
- start with a straight line
    - no activation, no bias
    - chain rule and gradient checking

In [1]:
'''
start with a straight line
suppose x = 2.0 , target y = 10.0 , current weight w = 3.0
1. Forward: y_hat = w * x  => y_hat = 3.0 * 2.0 = 6.0,
   squared error loss L = 0.5 * (y - y_hat)**2 => L = 0.5 * (10.0 - 6.0)**2 = 8.0

2. Backward(chain rule):
   dL/dy_hat = y_hat - y => dL/dy_hat = 6.0 - 10.0 = -4.0 : how much the loss changes if y_hat goes up.
   dy_hat/dw = x => dy_hat/dw = 2.0 : how much y_hat changes if w goes up.
   dL/dw = dL/dy_hat * dy_hat/dw => dL/dw = -4.0 * 2.0 = -8.0 : -ve means increasing w will decrease the loss.
'''

# Simple 1-weight linear example: L = 0.5 * (w * x - y)^2
x_simple = 2.0
y_simple = 10.0
w_simple = 3.0

# Forward
y_hat_simple = w_simple * x_simple
loss_simple = 0.5 * (y_hat_simple - y_simple) ** 2

# Backward (chain rule)
dL_dyhat_s = y_hat_simple - y_simple
dyhat_dw_s = x_simple
dL_dw_analytic_s = dL_dyhat_s * dyhat_dw_s

# Numerical check (nudge w slightly up and down)
eps = 1e-7 # epsilon for numerical gradient calculation
loss_plus = 0.5 * ((w_simple + eps) * x_simple - y_simple) ** 2
loss_minus = 0.5 * ((w_simple - eps) * x_simple - y_simple) ** 2
dL_dw_num_s = (loss_plus - loss_minus) / (2 * eps)

print(f"Simple Loss: {loss_simple:.4f}")
print(f"Simple dL/dw -> analytic: {dL_dw_analytic_s:.6f}, numerical: {dL_dw_num_s:.6f}")


Simple Loss: 8.0000
Simple dL/dw -> analytic: -8.000000, numerical: -8.000000


Backpropagation is **reverse-mode automatic differentiation** applied to a loss function $\mathcal{L}(\hat{y}, y)$ over a computation graph. It systematically applies the **chain rule** from the output loss backward to every parameter ($W, b$) so we don't compute expensive numerical finite-difference gradients for millions of parameters one by one.

- **Forward Pass:** Compute intermediate activations and pre-activations layer by layer from input $x$ to loss $\mathcal{L}$. Cache needed forward values (such as $x$, $z$, or post-activation outputs) for the backward pass.
- **Backward Pass (Chain Rule):** Start with $\frac{\partial \mathcal{L}}{\partial \hat{y}}$ and propagate sensitivities ("errors" / adjoints) backward through each activation and linear operation:
  - For a scalar loss $\mathcal{L}$ and composition $a = g(z)$, $z = w^\top x + b$:
    $$\frac{\partial \mathcal{L}}{\partial z} = \frac{\partial \mathcal{L}}{\partial a} \cdot g'(z), \quad \frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial z} x, \quad \frac{\partial \mathcal{L}}{\partial b} = \frac{\partial \mathcal{L}}{\partial z}, \quad \frac{\partial \mathcal{L}}{\partial x} = W^\top \frac{\partial \mathcal{L}}{\partial z}$$
- **Parameter Update (Gradient Descent):** $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$, where $\eta$ is the learning rate.
- **Key Edge Cases in Backpropagation:**
  1. *Vanishing Gradients:* Multiplying many small derivatives (e.g., sigmoid derivative $\le 0.25$ across deep layers) shrinks gradients exponentially toward $0$, stalling early layers.
  2. *Exploding Gradients:* Large weights or high learning rates can blow up gradients to `inf` or `NaN`.
  3. *Non-differentiable kinks (e.g., ReLU at $z = 0$):* In practice, floating-point values rarely hit exact `0.0`, and subgradients (`0` or `1`) work cleanly in standard implementations.
  4. *Stale/missing cache or wrong shapes:* Always verify matrix dimensions (`(in_features, out_features)` vs `(out_features, in_features)`) and batch reduction (mean vs. sum over batch size $N$).

In [3]:
# Toy 1-neuron MSE gradient check: L = 0.5 * (y_hat - y)^2, y_hat = sigmoid(w*x + b)
x_val = 1.2
y_true = 1.0
w_val = -0.5
b_val = 0.1
import numpy as np
def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z, dtype=np.float64)
    pos_mask = z >= 0
    neg_mask = ~pos_mask
    out[pos_mask] = 1.0 / (1.0 + np.exp(-z[pos_mask]))
    exp_z = np.exp(z[neg_mask])
    out[neg_mask] = exp_z / (1.0 + exp_z)
    return out

# Forward
z_val = w_val * x_val + b_val
y_hat = float(sigmoid(z_val))
loss = 0.5 * (y_hat - y_true) ** 2

# Backward (Analytic)
dL_dyhat = y_hat - y_true # step 1 : derivative of loss w.r.t y_hat : dL/dy_hat = y_hat - y_true
dyhat_dz = y_hat * (1.0 - y_hat) # step 2 : derivative of y_hat w.r.t z : dy_hat/dz = y_hat * (1 - y_hat)
dL_dz = dL_dyhat * dyhat_dz # step 3 : derivative of loss w.r.t z : dL/dz = dL/dy_hat * dy_hat/dz
dL_dw_analytic = dL_dz * x_val # step 4 : derivative of loss w.r.t w : dL/dw = dL/dz * x
dL_db_analytic = dL_dz * 1.0 # step 5 : derivative of loss w.r.t b : dL/db = dL/dz * 1

# Numerical gradient check (centered finite difference)
eps = 1e-7
def loss_from_wb(w_p, b_p): # compute loss for given w and b
    yh = float(sigmoid(w_p * x_val + b_p))
    return 0.5 * (yh - y_true) ** 2

dL_dw_num = (loss_from_wb(w_val + eps, b_val) - loss_from_wb(w_val - eps, b_val)) / (2 * eps)
dL_db_num = (loss_from_wb(w_val, b_val + eps) - loss_from_wb(w_val, b_val - eps)) / (2 * eps)


print(f"Loss: {loss:.6f}")
print(f"dL/dw -> analytic: {dL_dw_analytic:.8f}, numerical: {dL_dw_num:.8f}, diff: {abs(dL_dw_analytic - dL_dw_num):.2e}")
print(f"dL/db -> analytic: {dL_db_analytic:.8f}, numerical: {dL_db_num:.8f}, diff: {abs(dL_db_analytic - dL_db_num):.2e}")

Loss: 0.193728
dL/dw -> analytic: -0.17553630, numerical: -0.17553630, diff: 2.68e-10
dL/db -> analytic: -0.14628025, numerical: -0.14628025, diff: 6.12e-11
